# Stage 6 & 7 — Model Development and Optimization: Gradient Boosting

## Online Shopper Purchase Likelihood Prediction System

**IT3051 – Fundamentals of Data Mining | Mini Project 2026 | Group DS_WE_01.02**

**Member 4 — Gradient Boosting**

This notebook combines Stage 6 (Model Development) and Stage 7 (Optimization)
for the assigned Gradient Boosting algorithm.

The experiments follow the group's shared modelling setup:

- Selected training features from the feature-selection stage
- 5-fold stratified cross-validation
- Preprocessing fitted inside each cross-validation fold
- Purchase-class F1 as the main metric
- Precision, recall, ROC-AUC and PR-AUC also reported
- No test-set access during model development
- Balanced sample weighting tested for class imbalance
- Hyperparameter tuning using cross-validation
- Comparison with and without engineered features
- Comparison with and without `PageValues`

The final test set remains completely untouched until the group selects the
final model and performs final evaluation.

# 18. Final Summary — Gradient Boosting

The Gradient Boosting model was developed using five-fold stratified
cross-validation on the selected training data.

The modelling process included:

1. Untuned Gradient Boosting baseline
2. Balanced sample weighting
3. Randomized hyperparameter tuning
4. Tuned vs. untuned comparison
5. Engineered-feature comparison
6. `PageValues` comparison
7. Recording all experimental results

The candidate Gradient Boosting configuration uses the tuned hyperparameters
found by the randomized search, balanced sample weighting, the selected
features, engineered features, and `PageValues`.

The reported performance is based entirely on cross-validation of the training
data.

**The test set was not used in any experiment in this notebook.**

The test set should only be used after the group has completed the model
comparison and selected the final model.

In [22]:
import os
import sys
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    RandomizedSearchCV,
)

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.dummy import DummyClassifier

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.utils.class_weight import compute_sample_weight
from sklearn.base import BaseEstimator, ClassifierMixin, clone

from scipy.stats import randint, uniform

sys.path.insert(0, os.path.abspath(".."))

from src.preprocessing.encoding_scaling import (
    CATEGORICAL_COLUMNS,
    build_preprocessor,
)

RANDOM_STATE = 42

## 1. Load the selected training data

The model-development experiments use `X_train_selected.csv`, which is the
output of the group's feature-selection stage.

The test set is deliberately not loaded.

Preprocessing is still performed inside each cross-validation fold to prevent
data leakage.

In [23]:
X_train = pd.read_csv(
    "../data/processed/X_train_selected.csv"
)

y_train = pd.read_csv(
    "../data/y_train.csv"
).squeeze("columns").astype(int)

assert len(X_train) == len(y_train), (
    "X_train and y_train have different numbers of rows."
)

assert "Revenue" not in X_train.columns, (
    "Revenue must not be included in the input features."
)

print("Training rows:", len(X_train))
print("Training columns:", X_train.shape[1])
print("Purchase rate: {:.3f}".format(y_train.mean()))
print("Purchase count:", int(y_train.sum()))
print("Non-purchase count:", int((y_train == 0).sum()))

Training rows: 9764
Training columns: 18
Purchase rate: 0.156
Purchase count: 1526
Non-purchase count: 8238


## 2. Identify selected categorical and numerical features

The selected dataset contains only the features retained by the feature-selection
stage.

The categorical columns are taken from the shared preprocessing module and
restricted to columns that are actually present in the selected dataset.

In [24]:
selected_categorical_columns = [
    column
    for column in CATEGORICAL_COLUMNS
    if column in X_train.columns
]

selected_numeric_columns = [
    column
    for column in X_train.columns
    if column not in selected_categorical_columns
]

print("Selected categorical features:")
print(selected_categorical_columns)

print("\nSelected numerical features:")
print(selected_numeric_columns)

print("\nNumber of categorical features:", len(selected_categorical_columns))
print("Number of numerical features:", len(selected_numeric_columns))
print("Total selected features:", len(X_train.columns))

Selected categorical features:
['Month', 'OperatingSystems', 'Browser', 'TrafficType', 'VisitorType', 'Weekend', 'VisitorType_Weekend']

Selected numerical features:
['Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration', 'BounceRates', 'ExitRates', 'PageValues', 'SpecialDay', 'TotalPages', 'TotalDuration', 'AvgTimePerPage']

Number of categorical features: 7
Number of numerical features: 11
Total selected features: 18


In [25]:
print("\nFeature list:")
for i, column in enumerate(X_train.columns, start=1):
    print(f"{i:2}. {column}")


Feature list:
 1. Administrative
 2. Administrative_Duration
 3. Informational
 4. Informational_Duration
 5. BounceRates
 6. ExitRates
 7. PageValues
 8. SpecialDay
 9. Month
10. OperatingSystems
11. Browser
12. TrafficType
13. VisitorType
14. Weekend
15. TotalPages
16. TotalDuration
17. AvgTimePerPage
18. VisitorType_Weekend


## 3. Shared cross-validation setup

All Gradient Boosting experiments use the same 5-fold stratified
cross-validation configuration so that the results are directly comparable.

F1 is the main comparison metric because the target class is imbalanced.
ROC-AUC and PR-AUC are also reported to evaluate ranking/separation ability.

In [26]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

SCORING = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
}


def summarise(scores):
    """
    Convert cross_validate output into mean and standard deviation
    for each evaluation metric.
    """
    row = {}

    for metric in SCORING:
        values = scores["test_" + metric]

        row[metric + "_mean"] = values.mean()
        row[metric + "_std"] = values.std()

    return row

In [27]:
dummy_pipeline = Pipeline([
    (
        "prep",
        build_preprocessor(
            selected_numeric_columns,
            selected_categorical_columns,
        ),
    ),
    (
        "model",
        DummyClassifier(strategy="most_frequent"),
    ),
])

with warnings.catch_warnings():
    warnings.simplefilter("ignore")

    dummy_scores = cross_validate(
        dummy_pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=SCORING,
    )

dummy_result = summarise(dummy_scores)

print("Always-predict-no-purchase baseline:")
for metric, value in dummy_result.items():
    print(f"{metric}: {value:.4f}")

Always-predict-no-purchase baseline:
f1_mean: 0.0000
f1_std: 0.0000
precision_mean: 0.0000
precision_std: 0.0000
recall_mean: 0.0000
recall_std: 0.0000
roc_auc_mean: 0.5000
roc_auc_std: 0.0000
pr_auc_mean: 0.1563
pr_auc_std: 0.0002


## 5. Gradient Boosting

Gradient Boosting builds an ensemble of decision trees sequentially.

Instead of training many independent trees and combining their predictions,
each new tree is trained to reduce errors made by the previous trees.

The contribution of each tree is controlled by the learning rate.

This makes Gradient Boosting suitable for structured/tabular classification
problems where relationships between variables may be non-linear.

# Stage 6 — Model Development

## 6. Untuned Gradient Boosting

The first model uses the default `GradientBoostingClassifier` configuration.

This provides the reference result against which the optimization experiments
in Stage 7 will be compared.

In [28]:
gb_pipeline = Pipeline([
    (
        "prep",
        build_preprocessor(
            selected_numeric_columns,
            selected_categorical_columns,
        ),
    ),
    (
        "model",
        GradientBoostingClassifier(
            random_state=RANDOM_STATE
        ),
    ),
])

gb_scores = cross_validate(
    gb_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring=SCORING,
)

result_plain = summarise(gb_scores)

print("Untuned Gradient Boosting:")
for metric, value in result_plain.items():
    print(f"{metric}: {value:.4f}")

Untuned Gradient Boosting:
f1_mean: 0.6488
f1_std: 0.0224
precision_mean: 0.7219
precision_std: 0.0297
recall_mean: 0.5898
recall_std: 0.0245
roc_auc_mean: 0.9293
roc_auc_std: 0.0065
pr_auc_mean: 0.7419
pr_auc_std: 0.0234


In [29]:
stage6_results = pd.DataFrame([
    {
        "configuration": "Baseline - always no purchase",
        **dummy_result,
    },
    {
        "configuration": "Gradient Boosting - untuned",
        **result_plain,
    },
])

stage6_results.round(4)

,configuration,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,Baseline - always no purchase,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.5000,0.0000,0.1563,0.0002
1,Gradient Boosting - untuned,0.6488,0.0224,0.7219,0.0297,0.5898,0.0245,0.9293,0.0065,0.7419,0.0234


## 7. Stage 6 observation

The untuned Gradient Boosting model provides the reference point for Stage 7.

The main purpose of this result is not to claim that the default configuration
is optimal, but to establish a reproducible baseline before applying:

1. Balanced sample weighting
2. Hyperparameter tuning
3. Feature comparisons
4. `PageValues` comparison

All later experiments use the same cross-validation structure.

# Stage 7 — Optimization

## 8. Balanced sample weighting

`GradientBoostingClassifier` does not provide a `class_weight="balanced"`
parameter like some other classifiers.

Instead, balanced sample weights are calculated from the training labels
and passed to the classifier through `sample_weight`.

The wrapper below calculates the weights inside each fitted training fold.
Therefore, the weighting does not use labels from the validation fold.

In [30]:
class BalancedSampleWeightClassifier(
    ClassifierMixin,
    BaseEstimator
):
    """
    Wrap a classifier that supports sample_weight but does not
    provide class_weight.

    Balanced sample weights are calculated from the training
    labels supplied to each fit operation.
    """

    def __init__(self, estimator=None):
        self.estimator = estimator

    def fit(self, X, y, **kwargs):
        self.estimator_ = clone(self.estimator)

        sample_weight = compute_sample_weight(
            class_weight="balanced",
            y=y,
        )

        self.estimator_.fit(
            X,
            y,
            sample_weight=sample_weight,
        )

        self.classes_ = self.estimator_.classes_

        return self

    def predict(self, X):
        return self.estimator_.predict(X)

    def predict_proba(self, X):
        return self.estimator_.predict_proba(X)

## 9. Step 1 — Untuned, unweighted model

This reproduces the Stage 6 model inside the Stage 7 notebook so that all
comparisons are available in one place.

In [31]:
pipe_plain = Pipeline([
    (
        "prep",
        build_preprocessor(
            selected_numeric_columns,
            selected_categorical_columns,
        ),
    ),
    (
        "model",
        GradientBoostingClassifier(
            random_state=RANDOM_STATE
        ),
    ),
])

scores_plain = cross_validate(
    pipe_plain,
    X_train,
    y_train,
    cv=cv,
    scoring=SCORING,
)

result_plain = summarise(scores_plain)

print("Untuned, unweighted:")
for metric, value in result_plain.items():
    print(f"{metric}: {value:.4f}")

Untuned, unweighted:
f1_mean: 0.6488
f1_std: 0.0224
precision_mean: 0.7219
precision_std: 0.0297
recall_mean: 0.5898
recall_std: 0.0245
roc_auc_mean: 0.9293
roc_auc_std: 0.0065
pr_auc_mean: 0.7419
pr_auc_std: 0.0234


## 10. Step 2 — Untuned Gradient Boosting with balanced weighting

The next experiment keeps the Gradient Boosting hyperparameters unchanged
and changes only the class-imbalance treatment.

This allows the effect of balanced weighting to be examined separately.

In [32]:
pipe_weighted = Pipeline([
    (
        "prep",
        build_preprocessor(
            selected_numeric_columns,
            selected_categorical_columns,
        ),
    ),
    (
        "model",
        BalancedSampleWeightClassifier(
            GradientBoostingClassifier(
                random_state=RANDOM_STATE
            )
        ),
    ),
])

scores_weighted = cross_validate(
    pipe_weighted,
    X_train,
    y_train,
    cv=cv,
    scoring=SCORING,
)

result_weighted = summarise(scores_weighted)

print("Untuned + balanced weighting:")
for metric, value in result_weighted.items():
    print(f"{metric}: {value:.4f}")

KeyboardInterrupt: 

In [ ]:
weighting_comparison = pd.DataFrame([
    {
        "configuration": "Untuned + unweighted",
        **result_plain,
    },
    {
        "configuration": "Untuned + balanced weighting",
        **result_weighted,
    },
])

weighting_comparison.round(4)

,configuration,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,Untuned + unweighted,0.6488,0.0224,0.7219,0.0297,0.5898,0.0245,0.9293,0.0065,0.7419,0.0234
1,Untuned + balanced weighting,0.6510,0.0117,0.5372,0.0109,0.8263,0.0206,0.9302,0.0063,0.7372,0.0215


## Observation — Balanced weighting

Balanced weighting is evaluated based on the observed precision/recall/F1
trade-off.

If weighting substantially increases recall while precision decreases, this
indicates that the model is identifying more purchase cases but also producing
more false positives.

The configuration is carried forward for tuning so that the effect of
hyperparameter optimization can be evaluated under the same imbalance
treatment.

## 11. Step 3 — Hyperparameter tuning

A small randomized search is used to keep the computational cost practical.

The following important Gradient Boosting parameters are tuned:

- `n_estimators`
- `learning_rate`
- `max_depth`

The balanced weighting configuration is kept fixed during tuning.

The search is optimized using F1 because F1 is the project's primary model
comparison metric.

In [ ]:
param_distributions = {
    "model__estimator__n_estimators": randint(50, 200),
    "model__estimator__learning_rate": uniform(0.03, 0.22),
    "model__estimator__max_depth": randint(2, 5),
}

pipe_for_search = Pipeline([
    (
        "prep",
        build_preprocessor(
            selected_numeric_columns,
            selected_categorical_columns,
        ),
    ),
    (
        "model",
        BalancedSampleWeightClassifier(
            GradientBoostingClassifier(
                random_state=RANDOM_STATE
            )
        ),
    ),
])

search = RandomizedSearchCV(
    estimator=pipe_for_search,
    param_distributions=param_distributions,
    n_iter=10,
    scoring="f1",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

search.fit(X_train, y_train)

best_params = {
    key.replace("model__estimator__", ""): value
    for key, value in search.best_params_.items()
}

print("Best parameters:")
print(best_params)

print("\nBest CV F1 during search:")
print(round(search.best_score_, 4))

Best parameters:
{'learning_rate': np.float64(0.03017132848502315), 'max_depth': 2, 'n_estimators': 107}

Best CV F1 during search:
0.662


### Tuning limitation

Only 10 random parameter combinations are evaluated.

This is a deliberate computational trade-off for the project rather than an
exhaustive search. A larger search could potentially identify another
configuration, but the selected result is based on the search budget used here.

## 12. Step 4 — Tuned + balanced Gradient Boosting

The best hyperparameters from the randomized search are now evaluated using
the complete metric set.

The same five-fold cross-validation configuration is retained.

In [ ]:
tuned_model = GradientBoostingClassifier(
    random_state=RANDOM_STATE,
    **best_params,
)

pipe_tuned = Pipeline([
    (
        "prep",
        build_preprocessor(
            selected_numeric_columns,
            selected_categorical_columns,
        ),
    ),
    (
        "model",
        BalancedSampleWeightClassifier(
            tuned_model
        ),
    ),
])

scores_tuned = cross_validate(
    pipe_tuned,
    X_train,
    y_train,
    cv=cv,
    scoring=SCORING,
)

result_tuned = summarise(scores_tuned)

print("Tuned + balanced weighting:")
for metric, value in result_tuned.items():
    print(f"{metric}: {value:.4f}")

Tuned + balanced weighting:
f1_mean: 0.6620
f1_std: 0.0126
precision_mean: 0.5638
precision_std: 0.0157
recall_mean: 0.8027
recall_std: 0.0280
roc_auc_mean: 0.9241
roc_auc_std: 0.0062
pr_auc_mean: 0.7196
pr_auc_std: 0.0258


In [ ]:
comparison_untuned_tuned = pd.DataFrame([
    {
        "configuration": "Untuned + balanced weighting",
        **result_weighted,
    },
    {
        "configuration": "Tuned + balanced weighting",
        **result_tuned,
    },
])

comparison_untuned_tuned["f1_delta"] = (
    comparison_untuned_tuned["f1_mean"]
    - result_weighted["f1_mean"]
)

comparison_untuned_tuned.round(4)

,configuration,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std,f1_delta
0,Untuned + balanced weighting,0.651,0.0117,0.5372,0.0109,0.8263,0.0206,0.9302,0.0063,0.7372,0.0215,0.000
1,Tuned + balanced weighting,0.662,0.0126,0.5638,0.0157,0.8027,0.0280,0.9241,0.0062,0.7196,0.0258,0.011


## 13. Step 5 — With vs. without engineered features

The engineered features considered by the project are:

- `TotalPages`
- `TotalDuration`
- `AvgTimePerPage`
- `VisitorType_Weekend`

The tuned hyperparameters and balanced weighting remain fixed.

Only the engineered features are removed.

This allows us to determine whether these explicitly engineered features provide
additional value for Gradient Boosting.

In [ ]:
# ============================================================
# Engineered-Feature Comparison
# ============================================================
#
# 21 features = 17 original logged features + 4 engineered features
# 17 features = original logged features only
# 18 features = selected features from Member 3
#
# The 21 vs 17 comparison measures the contribution of
# engineered features.
# The 18-feature configuration represents feature selection.
# ============================================================

engineered_columns = [
    "TotalPages",
    "TotalDuration",
    "AvgTimePerPage",
    "VisitorType_Weekend",
]

# Load the 21-feature engineered training dataset
X_train_engineered = pd.read_csv(
    "../data/processed/X_train_engineered.csv"
)

# ------------------------------------------------------------
# 21-feature dataset: original logged + engineered features
# ------------------------------------------------------------

X_train_21 = X_train_engineered.copy()

# ------------------------------------------------------------
# 17-feature dataset: original logged features only
# ------------------------------------------------------------

X_train_17 = X_train_21.drop(
    columns=engineered_columns
)

# ------------------------------------------------------------
# 18-feature dataset: selected features from Member 3
# ------------------------------------------------------------

X_train_18 = X_train.copy()

print("Feature counts:")
print("Original logged features (17):", X_train_17.shape[1])
print("Original + engineered features (21):", X_train_21.shape[1])
print("Selected features (18):", X_train_18.shape[1])


# ------------------------------------------------------------
# Build preprocessing based on each feature set
# ------------------------------------------------------------

def build_comparison_preprocessor(X):
    categorical_cols = [
        column
        for column in X.columns
        if column in CATEGORICAL_COLUMNS
    ]

    numerical_cols = [
        column
        for column in X.columns
        if column not in categorical_cols
    ]

    return build_preprocessor(
        numerical_cols,
        categorical_cols,
    )


# ------------------------------------------------------------
# 21 features: original + engineered
# ------------------------------------------------------------

pipe_21 = Pipeline([
    (
        "prep",
        build_comparison_preprocessor(X_train_21),
    ),
    (
        "model",
        BalancedSampleWeightClassifier(
            GradientBoostingClassifier(
                random_state=RANDOM_STATE,
                **best_params,
            )
        ),
    ),
])

scores_21 = cross_validate(
    pipe_21,
    X_train_21,
    y_train,
    cv=cv,
    scoring=SCORING,
)

result_21 = summarise(scores_21)


# ------------------------------------------------------------
# 17 features: original logged features only
# ------------------------------------------------------------

pipe_17 = Pipeline([
    (
        "prep",
        build_comparison_preprocessor(X_train_17),
    ),
    (
        "model",
        BalancedSampleWeightClassifier(
            GradientBoostingClassifier(
                random_state=RANDOM_STATE,
                **best_params,
            )
        ),
    ),
])

scores_17 = cross_validate(
    pipe_17,
    X_train_17,
    y_train,
    cv=cv,
    scoring=SCORING,
)

result_17 = summarise(scores_17)


# ------------------------------------------------------------
# 18 features: selected features
# ------------------------------------------------------------

pipe_18 = Pipeline([
    (
        "prep",
        build_comparison_preprocessor(X_train_18),
    ),
    (
        "model",
        BalancedSampleWeightClassifier(
            GradientBoostingClassifier(
                random_state=RANDOM_STATE,
                **best_params,
            )
        ),
    ),
])

scores_18 = cross_validate(
    pipe_18,
    X_train_18,
    y_train,
    cv=cv,
    scoring=SCORING,
)

result_18 = summarise(scores_18)


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n21 features — original + engineered:")
for metric, value in result_21.items():
    print(f"{metric}: {value:.4f}")

print("\n17 features — original logged features:")
for metric, value in result_17.items():
    print(f"{metric}: {value:.4f}")

print("\n18 features — selected features:")
for metric, value in result_18.items():
    print(f"{metric}: {value:.4f}")


# ------------------------------------------------------------
# Final comparison table
# ------------------------------------------------------------

comparison_features = pd.DataFrame([
    {
        "configuration": "Tuned, original + engineered features (21)",
        **result_21,
    },
    {
        "configuration": "Tuned, original logged features (17)",
        **result_17,
    },
    {
        "configuration": "Tuned, selected features (18)",
        **result_18,
    },
])

comparison_features.round(4)

Feature counts:
Original logged features (17): 17
Original + engineered features (21): 21
Selected features (18): 18

21 features — original + engineered:
f1_mean: 0.6620
f1_std: 0.0126
precision_mean: 0.5638
precision_std: 0.0157
recall_mean: 0.8027
recall_std: 0.0280
roc_auc_mean: 0.9244
roc_auc_std: 0.0059
pr_auc_mean: 0.7196
pr_auc_std: 0.0245

17 features — original logged features:
f1_mean: 0.6620
f1_std: 0.0124
precision_mean: 0.5638
precision_std: 0.0153
recall_mean: 0.8027
recall_std: 0.0280
roc_auc_mean: 0.9241
roc_auc_std: 0.0056
pr_auc_mean: 0.7167
pr_auc_std: 0.0235

18 features — selected features:
f1_mean: 0.6620
f1_std: 0.0126
precision_mean: 0.5638
precision_std: 0.0157
recall_mean: 0.8027
recall_std: 0.0280
roc_auc_mean: 0.9241
roc_auc_std: 0.0062
pr_auc_mean: 0.7196
pr_auc_std: 0.0258


,configuration,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,"Tuned, original + engineered features (21)",0.662,0.0126,0.5638,0.0157,0.8027,0.028,0.9244,0.0059,0.7196,0.0245
1,"Tuned, original logged features (17)",0.662,0.0124,0.5638,0.0153,0.8027,0.028,0.9241,0.0056,0.7167,0.0235
2,"Tuned, selected features (18)",0.662,0.0126,0.5638,0.0157,0.8027,0.028,0.9241,0.0062,0.7196,0.0258


## 14. Step 6 — With vs. without `PageValues`

The tuned, balanced-weighted configuration is retained.

Only `PageValues` is removed.

This experiment is important because the earlier preprocessing experiments
also showed a substantial performance difference when `PageValues` was removed.

A large performance difference does not by itself establish whether
`PageValues` is appropriate for deployment. Its availability at the intended
prediction time must still be checked against the dataset documentation and
the project's prediction scenario.

In [ ]:
assert "PageValues" in X_train.columns, (
    "PageValues is not present in the selected training data."
)

numeric_without_pagevalues = [
    column
    for column in selected_numeric_columns
    if column != "PageValues"
]

X_train_without_pagevalues = X_train.drop(
    columns=["PageValues"]
)

print("Features before:", X_train.shape[1])
print(
    "Features after removing PageValues:",
    X_train_without_pagevalues.shape[1]
)

Features before: 18
Features after removing PageValues: 17


In [ ]:
tuned_model_no_pv = GradientBoostingClassifier(
    random_state=RANDOM_STATE,
    **best_params,
)

pipe_no_pagevalues = Pipeline([
    (
        "prep",
        build_preprocessor(
            numeric_without_pagevalues,
            selected_categorical_columns,
        ),
    ),
    (
        "model",
        BalancedSampleWeightClassifier(
            tuned_model_no_pv
        ),
    ),
])

scores_no_pagevalues = cross_validate(
    pipe_no_pagevalues,
    X_train_without_pagevalues,
    y_train,
    cv=cv,
    scoring=SCORING,
)

result_no_pagevalues = summarise(
    scores_no_pagevalues
)

print("Tuned + balanced weighting, WITHOUT PageValues:")

for metric, value in result_no_pagevalues.items():
    print(f"{metric}: {value:.4f}")

Tuned + balanced weighting, WITHOUT PageValues:
f1_mean: 0.3897
f1_std: 0.0137
precision_mean: 0.2615
precision_std: 0.0097
recall_mean: 0.7647
recall_std: 0.0267
roc_auc_mean: 0.7579
roc_auc_std: 0.0173
pr_auc_mean: 0.3522
pr_auc_std: 0.0247


In [ ]:
comparison_pagevalues = pd.DataFrame([
    {
        "configuration": "With PageValues",
        **result_tuned,
    },
    {
        "configuration": "Without PageValues",
        **result_no_pagevalues,
    },
])

comparison_pagevalues.round(4)

,configuration,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,With PageValues,0.6620,0.0126,0.5638,0.0157,0.8027,0.0280,0.9241,0.0062,0.7196,0.0258
1,Without PageValues,0.3897,0.0137,0.2615,0.0097,0.7647,0.0267,0.7579,0.0173,0.3522,0.0247


In [ ]:
overall_results = pd.DataFrame([
    {
        "configuration": "Untuned, unweighted",
        **result_plain,
    },
    {
        "configuration": "Untuned + balanced weighting",
        **result_weighted,
    },
    {
        "configuration": "Tuned + balanced weighting",
        **result_tuned,
    },
    {
        "configuration": "Tuned, original logged features (17)",
        **result_17,
    },
    {
        "configuration": "Tuned, without PageValues",
        **result_no_pagevalues,
    },
])

overall_results.round(4)

,configuration,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,roc_auc_mean,roc_auc_std,pr_auc_mean,pr_auc_std
0,"Untuned, unweighted",0.6488,0.0224,0.7219,0.0297,0.5898,0.0245,0.9293,0.0065,0.7419,0.0234
1,Untuned + balanced weighting,0.6510,0.0117,0.5372,0.0109,0.8263,0.0206,0.9302,0.0063,0.7372,0.0215
2,Tuned + balanced weighting,0.6620,0.0126,0.5638,0.0157,0.8027,0.0280,0.9241,0.0062,0.7196,0.0258
3,"Tuned, original logged features (17)",0.6620,0.0124,0.5638,0.0153,0.8027,0.0280,0.9241,0.0056,0.7167,0.0235
4,"Tuned, without PageValues",0.3897,0.0137,0.2615,0.0097,0.7647,0.0267,0.7579,0.0173,0.3522,0.0247


In [ ]:
f1_comparison = overall_results[
    [
        "configuration",
        "f1_mean",
        "f1_std",
    ]
].copy()

f1_comparison.round(4)

,configuration,f1_mean,f1_std
0,"Untuned, unweighted",0.6488,0.0224
1,Untuned + balanced weighting,0.6510,0.0117
2,Tuned + balanced weighting,0.6620,0.0126
3,"Tuned, without engineered features",0.6622,0.0124
4,"Tuned, without PageValues",0.3897,0.0137


## 15. Gradient Boosting candidate configuration

Based on the cross-validation experiments, the candidate Gradient Boosting
configuration is:

- Tuned `n_estimators`
- Tuned `learning_rate`
- Tuned `max_depth`
- Balanced sample weighting
- Selected features
- Engineered features retained
- `PageValues` retained for the modelling comparison

The final decision to use `PageValues` for deployment should depend not only
on predictive performance but also on whether it is available at the intended
prediction point.

In [ ]:
print("Gradient Boosting candidate configuration")
print("=" * 50)

print("\nBest hyperparameters:")
for key, value in best_params.items():
    print(f"{key}: {value}")

print("\nFinal candidate metrics:")
for metric, value in result_tuned.items():
    print(f"{metric}: {value:.4f}")

Gradient Boosting candidate configuration

Best hyperparameters:
learning_rate: 0.03017132848502315
max_depth: 2
n_estimators: 107

Final candidate metrics:
f1_mean: 0.6620
f1_std: 0.0126
precision_mean: 0.5638
precision_std: 0.0157
recall_mean: 0.8027
recall_std: 0.0280
roc_auc_mean: 0.9241
roc_auc_std: 0.0062
pr_auc_mean: 0.7196
pr_auc_std: 0.0258


## 16. Interpretation

### Untuned vs tuned

The tuned configuration is compared directly with the untuned,
balanced-weighted configuration. The difference in F1 indicates whether the
limited hyperparameter search produced an improvement.

### Balanced weighting

Balanced sample weighting changes the precision-recall trade-off by giving
greater importance to the minority purchase class.

A substantial increase in recall accompanied by a decrease in precision means
that more actual purchasers are detected, but more non-purchasers are also
classified as potential purchasers.

### Engineered features

The comparison determines whether the explicitly engineered features provide
additional predictive value for Gradient Boosting.

Because decision trees can naturally model non-linear relationships and
interactions, engineered versions of information already represented by raw
features may provide limited additional benefit.

### PageValues

`PageValues` is evaluated separately because removing it can substantially
change predictive performance.

High predictive usefulness does not automatically mean that a feature is
safe for deployment. The project must verify that the feature is available
at the point where the prediction is intended to be made.

### Overall limitation

The hyperparameter search used only 10 random combinations. Therefore, the
identified configuration should be treated as the best configuration found
within the defined search budget, rather than as a globally optimal
configuration.

## 17. Save Gradient Boosting experiment results

The experiment results are saved so that the group's final comparison table
can be created without rerunning every experiment.

In [33]:
results_folder = "../results/member4_gradientboosting"

os.makedirs(
    results_folder,
    exist_ok=True,
)

In [34]:
baseline_df = pd.DataFrame([
    {
        "configuration": "Untuned, unweighted",
        **result_plain,
    }
])

baseline_df.to_csv(
    f"{results_folder}/baseline.csv",
    index=False,
)


weighting_df = pd.DataFrame([
    {
        "configuration": "Unweighted",
        **result_plain,
    },
    {
        "configuration": "Balanced weighting",
        **result_weighted,
    },
])

weighting_df.to_csv(
    f"{results_folder}/weighting_comparison.csv",
    index=False,
)


tuning_df = comparison_untuned_tuned.copy()

tuning_df.to_csv(
    f"{results_folder}/tuning_comparison.csv",
    index=False,
)


features_df = comparison_features.copy()

features_df.to_csv(
    f"{results_folder}/engineered_features_comparison.csv",
    index=False,
)


pagevalues_df = comparison_pagevalues.copy()

pagevalues_df.to_csv(
    f"{results_folder}/pagevalues_comparison.csv",
    index=False,
)


overall_results.to_csv(
    f"{results_folder}/overall_results.csv",
    index=False,
)

print("CSV results saved.")

CSV results saved.


In [ ]:
metadata = {
    "algorithm": "GradientBoostingClassifier",

    "data_source": "data/processed/X_train_selected.csv",

    "cross_validation": {
        "method": "StratifiedKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": RANDOM_STATE,
    },

    "scoring": [
        "f1",
        "precision",
        "recall",
        "roc_auc",
        "pr_auc",
    ],

    "best_hyperparameters": best_params,

    "weighting_method": (
        "balanced sample_weight computed inside each training fold"
    ),

    "final_candidate_configuration": (
        "tuned hyperparameters + balanced sample weighting "
        "+ selected features + engineered features + PageValues"
    ),

    "final_f1_mean": result_tuned["f1_mean"],

    "final_f1_std": result_tuned["f1_std"],

    "test_set_used": False,

    "random_search_iterations": 10,
}


with open(
    f"{results_folder}/candidate_metadata.json",
    "w"
) as f:
    json.dump(
        metadata,
        f,
        indent=2,
    )

print("Metadata saved.")

Metadata saved.


In [ ]:
from pathlib import Path
from joblib import dump

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

# A normal pipeline can store the fitted model without depending
# on a wrapper class defined inside the notebook.
candidate_pipeline = Pipeline([
    (
        "prep",
        build_preprocessor(
            selected_numeric_columns,
            selected_categorical_columns,
        ),
    ),
    (
        "model",
        GradientBoostingClassifier(
            random_state=RANDOM_STATE,
            **best_params,
        ),
    ),
])

training_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train,
)

candidate_pipeline.fit(
    X_train,
    y_train,
    model__sample_weight=training_weights,
)

models_folder = project_root / "models"
models_folder.mkdir(parents=True, exist_ok=True)

dump(
    candidate_pipeline,
    models_folder / "gradientboosting_candidate.joblib",
)

print("Saved Gradient Boosting candidate.")

Saved Gradient Boosting candidate.


# 18. Final Summary — Gradient Boosting

The Gradient Boosting model was developed using five-fold stratified
cross-validation on the selected training data.

The modelling process included:

1. Untuned Gradient Boosting baseline
2. Balanced sample weighting
3. Randomized hyperparameter tuning
4. Tuned vs. untuned comparison
5. Engineered-feature comparison
6. `PageValues` comparison
7. Recording all experimental results

The candidate Gradient Boosting configuration uses the tuned hyperparameters
found by the randomized search, balanced sample weighting, the selected
features, engineered features, and `PageValues`.

The reported performance is based entirely on cross-validation of the training
data.

**The test set was not used in any experiment in this notebook.**

The test set should only be used after the group has completed the model
comparison and selected the final model.